# Remote NDT — competency question test harness

Turns a filled-in data template into RDF, merges it with the RNDT ontology schema, and runs all
**34 competency questions** against it.

**To use it:** put your data in `rndt-data-template.xlsx` (start from `rndt-data-template-blank.xlsx`),
point the config cell below at it, then *Run All*. Every question reports pass/fail with the actual
answers it returned.

The template holds **instance data only** — your inspections, components, people, images. The taxonomy
(Process Stages, Major Actions, Action Areas, Activities, Authority Classes) lives in the schema modules —
`rndt-core.ttl` (generic, method-agnostic) plus one or more method extensions, e.g. `rndt-dr.ttl` for
Digital Radiography and `rndt-ut.ttl` for Ultrasonic Testing — and is merged in automatically, so you
never retype it and it cannot drift. This template currently carries one worked example of each method
side by side (INSP_001–009 for DR, INSP_010 for UT) against the same shared core. Method-specific data
lives on its own extension sheet, keyed by the same id as the base row it extends — approved-technique
data on `ProceduresDR` / `ProceduresUT`, per-inspection data on `InspectionsDR` / `InspectionsUT`, and
UT probe attributes on `EquipmentUT` — so a method's columns never sit blank on another method's row.

The populated data is written back out **one file per method**: `rndt-populated-dr.ttl`,
`rndt-populated-ut.ttl`, and `rndt-populated-shared.ttl` for the individuals both episodes refer to
(the people, the governing specification), so each method has a citable populated ontology of its own
without either file duplicating or dangling.

Input may be either the `.xlsx` **or** a directory of CSVs, one per sheet — both produce identical results.

*Requires only `rdflib` and `openpyxl`.*

In [1]:
# Setup — installs the two dependencies if they are missing, then loads helpers.
import sys, subprocess
for _pkg, _mod in [("rdflib", "rdflib"), ("openpyxl", "openpyxl")]:
    try:
        __import__(_mod)
    except ImportError:
        print(f"installing {_pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", _pkg])

import csv, os, re, html, json
from collections import defaultdict
from pathlib import Path
import rdflib
from rdflib import Literal, URIRef
from rdflib.namespace import RDF, RDFS, OWL, XSD
from IPython.display import HTML, display

RNDT = rdflib.Namespace("http://www.cranfield.ac.uk/rndt-ontology#")
RNDT_DR = rdflib.Namespace("http://www.cranfield.ac.uk/rndt-ontology/dr#")
RNDT_UT = rdflib.Namespace("http://www.cranfield.ac.uk/rndt-ontology/ut#")

_CSS = """
<style>
.rndt { font-family: ui-sans-serif, -apple-system, "Segoe UI", sans-serif; font-size: 13px;
        color: #1a1a18; margin: 4px 0 14px; }
.rndt .cap { font-size: 11px; text-transform: uppercase; letter-spacing: .06em;
             color: #8a8677; margin-bottom: 5px; font-weight: 600; }
.rndt table { border-collapse: collapse; background: #fff; border: 1px solid #e4e1d8;
              border-radius: 8px; overflow: hidden; }
.rndt th { text-align: left; font-size: 10px; text-transform: uppercase; letter-spacing: .05em;
           color: #8a8677; background: #f4f2ea; padding: 7px 11px; border-bottom: 1px solid #e4e1d8; }
.rndt td { padding: 6px 11px; border-bottom: 1px solid #efede4; color: #1a1a18;
           vertical-align: top; font-variant-numeric: tabular-nums; }
.rndt tr:last-child td { border-bottom: none; }
.rndt .mono { font-family: ui-monospace, "SF Mono", Consolas, monospace; font-size: 11.5px; }
.rndt .ok   { color: #1e6b30; font-weight: 600; }
.rndt .bad  { color: #a3341c; font-weight: 600; }
.rndt .warn { color: #8a6d3b; font-weight: 600; }
.rndt .dim  { color: #8a8677; }
.rndt pre { font-family: ui-monospace, "SF Mono", Consolas, monospace; font-size: 11px;
            background: #f8f7f2; border: 1px solid #ece9df; border-radius: 6px;
            padding: 8px 10px; margin: 5px 0; white-space: pre-wrap; color: #3a3934; }
.rndt .note { font-size: 12px; color: #57554c; margin: 4px 0; }
</style>
"""
_css_done = False

def table(headers, rows, caption=None, classes=None):
    """Render a small HTML table with the project styling."""
    global _css_done
    css = "" if _css_done else _CSS
    _css_done = True
    cls = classes or {}
    head = "".join(f"<th>{html.escape(str(h))}</th>" for h in headers)
    body = ""
    for r in rows:
        tds = ""
        for i, v in enumerate(r):
            c = cls.get(i, "")
            tds += f'<td class="{c}">{v if c.startswith("raw") else html.escape(str(v))}</td>'
        body += f"<tr>{tds}</tr>"
    cap = f'<div class="cap">{html.escape(caption)}</div>' if caption else ""
    return HTML(f'{css}<div class="rndt">{cap}<table><tr>{head}</tr>{body}</table></div>')

def note(msg, kind="note"):
    global _css_done
    css = "" if _css_done else _CSS
    _css_done = True
    return HTML(f'{css}<div class="rndt"><div class="{kind}">{msg}</div></div>')

print("ready —", "rdflib", rdflib.__version__)

ready — rdflib 7.6.0


In [2]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# The only cell you normally need to edit.

DATA_IN      = "rndt-data-template.xlsx"                             # .xlsx file, OR a directory of CSVs
SCHEMA_FILES = ["rndt-core.ttl", "rndt-dr.ttl", "rndt-ut.ttl"]        # generic core + method extension module(s) — merged in, never modified
TTL_OUT      = {                                                      # populated instance data, written one file per method
    "shared": "rndt-populated-shared.ttl",                            # individuals both methods' episodes refer to
    "DR":     "rndt-populated-dr.ttl",
    "UT":     "rndt-populated-ut.ttl",
}

# ──────────────────────────────────────────────────────────────────────────────
for _p in (DATA_IN, *SCHEMA_FILES):
    if not Path(_p).exists():
        raise FileNotFoundError(f"{_p!r} not found — check the path in this cell.")
print(f"input  : {DATA_IN}")
print(f"schema : {', '.join(SCHEMA_FILES)}")
print(f"output : {', '.join(TTL_OUT.values())}")

input  : rndt-data-template.xlsx
schema : rndt-core.ttl, rndt-dr.ttl, rndt-ut.ttl
output : rndt-populated-shared.ttl, rndt-populated-dr.ttl, rndt-populated-ut.ttl


In [3]:
# Load the template — accepts an .xlsx workbook or a directory of CSVs (one per sheet).

DATA_SHEETS = ["Components", "Participants", "Facilities", "Software", "Equipment", "EquipmentUT",
               "Procedures", "ProceduresDR", "ProceduresUT", "Specifications", "AcceptanceCriteria",
               "Inspections", "InspectionsDR", "InspectionsUT", "Images", "Reports", "Indications",
               "ActivityLog"]

def _clean(v):
    if v is None:
        return ""
    s = str(v).strip()
    return "" if s.lower() in ("none", "nan") else s

def load_xlsx(path):
    import openpyxl
    wb = openpyxl.load_workbook(path, data_only=True)
    out = {}
    for name in DATA_SHEETS:
        if name not in wb.sheetnames:
            out[name] = ([], [])
            continue
        ws = wb[name]
        rows = list(ws.iter_rows(values_only=True))
        if not rows:
            out[name] = ([], [])
            continue
        headers = [_clean(h) for h in rows[0] if _clean(h)]
        data = []
        for r in rows[1:]:
            vals = [_clean(v) for v in r[:len(headers)]]
            if any(vals):
                data.append(dict(zip(headers, vals)))
        out[name] = (headers, data)
    return out

def load_csvdir(path):
    out = {}
    for name in DATA_SHEETS:
        f = Path(path) / f"{name}.csv"
        if not f.exists():
            out[name] = ([], [])
            continue
        with open(f, newline="", encoding="utf-8-sig") as fh:
            rdr = list(csv.reader(fh))
        if not rdr:
            out[name] = ([], [])
            continue
        headers = [_clean(h) for h in rdr[0] if _clean(h)]
        data = []
        for r in rdr[1:]:
            vals = [_clean(v) for v in (list(r) + [""] * len(headers))[:len(headers)]]
            if any(vals):
                data.append(dict(zip(headers, vals)))
        out[name] = (headers, data)
    return out

DATA = load_csvdir(DATA_IN) if Path(DATA_IN).is_dir() else load_xlsx(DATA_IN)

_total = sum(len(v[1]) for v in DATA.values())
display(table(
    ["Sheet", "Rows", "Columns"],
    [[s, len(DATA[s][1]), ", ".join(DATA[s][0]) or "—"] for s in DATA_SHEETS],
    caption=f"Loaded {_total} data rows from {Path(DATA_IN).name}",
    classes={1: "mono"}))

Sheet,Rows,Columns
Components,10,"id, label, partNumber, serialNumber"
Participants,6,"id, name, role, authorityClass"
Facilities,3,"id, label"
Software,3,"id, label"
Equipment,4,"id, label, serialNumber"
EquipmentUT,1,"id, probeFrequency, probeDiameter, probeAngle"
Procedures,3,"id, label, revision, approvedBy"
ProceduresDR,2,"id, iqiType, radiographicTechnique, sourceToDetectorDistance, sourceToObjectDistance, requiredIQISensitivity"
ProceduresUT,1,"id, usesUTMethod"
Specifications,1,"id, label"


In [4]:
# Validate before converting. Problems are reported, not raised — a partly filled template
# still converts, and you simply see which competency questions it cannot answer.

schema_g = rdflib.Graph()
for _f in SCHEMA_FILES:
    schema_g.parse(_f, format="turtle")
_local = lambda u: str(u).split("#")[-1]
VOCAB_ACTIVITY  = {_local(s) for s in schema_g.subjects(RDF.type, RNDT.Activity)}
VOCAB_AUTHORITY = {_local(s) for s in schema_g.subjects(RDF.type, RNDT.AuthorityClass)}
VOCAB_IQITYPE   = {_local(s) for s in schema_g.subjects(RDF.type, RNDT_DR.IQIType)}
VOCAB_TECHNIQUE = {_local(s) for s in schema_g.subjects(RDF.type, RNDT_DR.RadiographicTechnique)}
VOCAB_UTMETHOD  = {_local(s) for s in schema_g.subjects(RDF.type, RNDT_UT.UTMethod)}
# name -> actual IRI, since these classes' individuals may come from more than one
# namespace (core vs a method extension like rndt-dr) — see add_vocab_ref below.
ACTIVITY_IRI  = {_local(s): s for s in schema_g.subjects(RDF.type, RNDT.Activity)}
IQITYPE_IRI   = {_local(s): s for s in schema_g.subjects(RDF.type, RNDT_DR.IQIType)}
TECHNIQUE_IRI = {_local(s): s for s in schema_g.subjects(RDF.type, RNDT_DR.RadiographicTechnique)}
UTMETHOD_IRI  = {_local(s): s for s in schema_g.subjects(RDF.type, RNDT_UT.UTMethod)}

# column -> sheet it must reference
REFS = {
    "Procedures":   {"approvedBy": "Participants"},
    "ProceduresDR": {"id": "Procedures"},   # extension row keyed by the same id as its base Procedure
    "ProceduresUT": {"id": "Procedures"},
    "EquipmentUT":   {"id": "Equipment"},    # extension row keyed by the same id as its base Equipment
    "InspectionsDR": {"id": "Inspections"},  # extension row keyed by the same id as its base Inspection
    "InspectionsUT": {"id": "Inspections", "probe": "Equipment"},
    "Inspections": {"component": "Components", "procedure": "Procedures",
                    "acceptanceCriteria": "AcceptanceCriteria", "equipment": "Equipment",
                    "specification": "Specifications", "facility": "Facilities"},
    "Images":      {"inspection": "Inspections", "derivedFrom": "Images",
                    "viewedWithSoftware": "Software"},
    "Reports":     {"inspection": "Inspections"},
    "Indications": {"image": "Images", "acceptanceCriteria": "AcceptanceCriteria"},
    "ActivityLog": {"inspection": "Inspections", "performedBy": "Participants"},
}
IDS = {s: {r.get("id", "") for r in DATA[s][1] if r.get("id")} for s in DATA_SHEETS}

issues = []
def flag(sheet, row, col, msg):
    issues.append([sheet, row, col, msg])

def _as_float(v):
    try:
        return float(v)
    except ValueError:
        return None

for sheet in DATA_SHEETS:
    headers, rows = DATA[sheet]
    if headers and "id" not in headers:
        flag(sheet, "—", "id", "sheet has no 'id' column")
    seen = {}
    for i, r in enumerate(rows, start=2):          # row 2 = first data row
        rid = r.get("id", "")
        if not rid:
            flag(sheet, i, "id", "missing id"); continue
        if not re.fullmatch(r"[A-Za-z0-9_\-]+", rid):
            flag(sheet, i, "id", f"invalid characters in id {rid!r} — use letters, digits, _ or -")
        if rid in seen:
            flag(sheet, i, "id", f"duplicate id {rid!r} (first seen on row {seen[rid]})")
        seen[rid] = i
        for col, target in REFS.get(sheet, {}).items():
            v = r.get(col, "")
            if v and v not in IDS[target]:
                flag(sheet, i, col, f"{v!r} is not an id in the {target} sheet")
        if sheet == "ActivityLog":
            a = r.get("activity", "")
            if a and a not in VOCAB_ACTIVITY:
                flag(sheet, i, "activity", f"{a!r} is not an Activity in {', '.join(SCHEMA_FILES)}")
        if sheet == "Participants":
            a = r.get("authorityClass", "")
            if a and a not in VOCAB_AUTHORITY:
                flag(sheet, i, "authorityClass", f"{a!r} is not an Authority Class in {', '.join(SCHEMA_FILES)}")
        if sheet == "ProceduresDR":
            iqi = r.get("iqiType", "")
            if iqi and iqi not in VOCAB_IQITYPE:
                flag(sheet, i, "iqiType", f"{iqi!r} is not an IQIType in {', '.join(SCHEMA_FILES)}")
            tech = r.get("radiographicTechnique", "")
            if tech and tech not in VOCAB_TECHNIQUE:
                flag(sheet, i, "radiographicTechnique", f"{tech!r} is not a RadiographicTechnique in {', '.join(SCHEMA_FILES)}")
            sdd, sod = _as_float(r.get("sourceToDetectorDistance", "")), _as_float(r.get("sourceToObjectDistance", ""))
            for col, val in (("sourceToDetectorDistance", sdd), ("sourceToObjectDistance", sod)):
                raw = r.get(col, "")
                if raw and val is None:
                    flag(sheet, i, col, f"{raw!r} is not a number")
            if sdd is not None and sod is not None and sdd <= sod:
                flag(sheet, i, "sourceToDetectorDistance",
                     f"SDD ({sdd}) must be greater than SOD ({sod}) — the object sits between source and detector")
        if sheet == "ProceduresUT":
            um = r.get("usesUTMethod", "")
            if um and um not in VOCAB_UTMETHOD:
                flag(sheet, i, "usesUTMethod", f"{um!r} is not a UTMethod in {', '.join(SCHEMA_FILES)}")
        if sheet == "ProceduresDR":
            raw = r.get("requiredIQISensitivity", "")
            if raw and _as_float(raw) is None:
                flag(sheet, i, "requiredIQISensitivity", f"{raw!r} is not a number")
        if sheet == "InspectionsDR":
            raw = r.get("iqiSensitivityAchieved", "")
            if raw and _as_float(raw) is None:
                flag(sheet, i, "iqiSensitivityAchieved", f"{raw!r} is not a number")
        if sheet == "EquipmentUT":
            for col in ("probeFrequency", "probeDiameter", "probeAngle"):
                raw = r.get(col, "")
                if raw and _as_float(raw) is None:
                    flag(sheet, i, col, f"{raw!r} is not a number")
        for col in ("date",):
            v = r.get(col, "")
            if v and not re.fullmatch(r"\d{4}-\d{2}-\d{2}", v):
                flag(sheet, i, col, f"{v!r} is not a YYYY-MM-DD date")

if issues:
    display(table(["Sheet", "Row", "Column", "Problem"], issues,
                  caption=f"{len(issues)} issue(s) found — these rows will be skipped or left incomplete",
                  classes={1: "mono", 2: "mono"}))
else:
    display(note("✓ No problems found — every id is unique and every reference resolves.", "ok"))

In [5]:
# Convert to RDF and write the Turtle file.

g = rdflib.Graph()
g.bind("rndt", RNDT); g.bind("rndt-dr", RNDT_DR); g.bind("rndt-ut", RNDT_UT); g.bind("rdfs", RDFS); g.bind("owl", OWL); g.bind("xsd", XSD)

U   = lambda i: RNDT[i]
def add_lit(s, p, v, dt=None):
    if v: g.add((U(s), p, Literal(v, datatype=dt) if dt else Literal(v)))
def add_ref(s, p, v, valid):
    if v and v in valid: g.add((U(s), p, U(v)))
def add_vocab_ref(s, p, v, name_to_iri):
    # Like add_ref, but resolves against a name -> full-IRI map rather than always
    # minting the target in the core rndt: namespace — needed for Activity, since
    # individuals of that class can come from the core module or a method
    # extension module (e.g. rndt-dr) that mints its own namespace.
    if v and v in name_to_iri: g.add((U(s), p, name_to_iri[v]))

def typed(s, cls, label=None):
    g.add((U(s), RDF.type, OWL.NamedIndividual)); g.add((U(s), RDF.type, RNDT[cls]))
    if label: g.add((U(s), RDFS.label, Literal(label, lang="en")))

def rows(sheet): return DATA[sheet][1]

for r in rows("Components"):
    typed(r["id"], "PhysicalEntity", r.get("label"))
    add_lit(r["id"], RNDT.partNumber, r.get("partNumber"))
    add_lit(r["id"], RNDT.serialNumber, r.get("serialNumber"))

for r in rows("Participants"):
    typed(r["id"], "Participant", r.get("name"))
    add_lit(r["id"], RNDT.hasRole, r.get("role"))
    add_ref(r["id"], RNDT.memberOfAuthorityClass, r.get("authorityClass"), VOCAB_AUTHORITY)

for r in rows("Facilities"):          typed(r["id"], "Facility", r.get("label"))
for r in rows("Software"):            typed(r["id"], "Software", r.get("label"))
for r in rows("Equipment"):
    typed(r["id"], "PhysicalEntity", r.get("label"))
    add_lit(r["id"], RNDT.serialNumber, r.get("serialNumber"))   # core property — equipment is a PhysicalEntity like any component

for r in rows("Specifications"):      typed(r["id"], "InformationEntity", r.get("label"))
for r in rows("AcceptanceCriteria"):  typed(r["id"], "InformationEntity", r.get("label"))

for r in rows("Procedures"):
    typed(r["id"], "Procedure", r.get("label"))
    g.add((U(r["id"]), RDF.type, RNDT.InformationEntity))   # Procedure is a subclass, but assert explicitly since nothing here runs an RDFS reasoner
    add_lit(r["id"], RNDT.procedureRevision, r.get("revision"))
    add_ref(r["id"], RNDT.approvedBy, r.get("approvedBy"), IDS["Participants"])

# Method-specific approved-technique data lives on its own extension sheet, keyed
# by the same id as the base Procedure row above — so these just add more triples
# onto an individual that's already been typed, rather than typing a new one.
for r in rows("ProceduresDR"):
    add_vocab_ref(r["id"], RNDT_DR.usesIQIType, r.get("iqiType"), IQITYPE_IRI)
    add_vocab_ref(r["id"], RNDT_DR.usesRadiographicTechnique, r.get("radiographicTechnique"), TECHNIQUE_IRI)
    add_lit(r["id"], RNDT_DR.sourceToDetectorDistance, r.get("sourceToDetectorDistance"), XSD.decimal)
    add_lit(r["id"], RNDT_DR.sourceToObjectDistance, r.get("sourceToObjectDistance"), XSD.decimal)

    add_lit(r["id"], RNDT_DR.requiredIQISensitivity, r.get("requiredIQISensitivity"), XSD.decimal)

for r in rows("ProceduresUT"):
    add_vocab_ref(r["id"], RNDT_UT.usesUTMethod, r.get("usesUTMethod"), UTMETHOD_IRI)

# Method-specific equipment data follows the same extension-sheet pattern. This one
# also narrows the type: a row here says the base Equipment individual is specifically
# a probe, so it gains rndt-ut:Probe alongside the rndt:PhysicalEntity it already has.
for r in rows("EquipmentUT"):
    g.add((U(r["id"]), RDF.type, RNDT_UT.Probe))
    add_lit(r["id"], RNDT_UT.probeFrequency, r.get("probeFrequency"), XSD.decimal)
    add_lit(r["id"], RNDT_UT.probeDiameter, r.get("probeDiameter"), XSD.decimal)
    add_lit(r["id"], RNDT_UT.probeAngle, r.get("probeAngle"), XSD.decimal)

for r in rows("Inspections"):
    typed(r["id"], "RemoteNDTProcess", r.get("label"))
    add_lit(r["id"], RNDT.inspectionDate, r.get("date"), XSD.date)
    add_lit(r["id"], RNDT.disposition, r.get("disposition"))
    add_ref(r["id"], RNDT.usesEntity, r.get("component"), IDS["Components"])
    add_ref(r["id"], RNDT.usesProcedure, r.get("procedure"), IDS["Procedures"])
    add_ref(r["id"], RNDT.usesAcceptanceCriteria, r.get("acceptanceCriteria"), IDS["AcceptanceCriteria"])
    add_ref(r["id"], RNDT.usesEquipment, r.get("equipment"), IDS["Equipment"])
    add_ref(r["id"], RNDT.appliesSpecification, r.get("specification"), IDS["Specifications"])
    add_ref(r["id"], RNDT.performedAtFacility, r.get("facility"), IDS["Facilities"])

# Per-inspection method-specific data, keyed by the base Inspection id — the same
# extension-sheet split the Procedures sheets use, so no method's column ever sits
# blank on another method's row.
for r in rows("InspectionsDR"):
    add_lit(r["id"], RNDT_DR.iqiSensitivityAchieved, r.get("iqiSensitivityAchieved"), XSD.decimal)

for r in rows("InspectionsUT"):
    add_ref(r["id"], RNDT_UT.usesProbe, r.get("probe"), IDS["Equipment"])

for r in rows("Images"):
    typed(r["id"], "InformationEntity", r.get("label"))
    add_ref(r["id"], RNDT.belongsToInspection, r.get("inspection"), IDS["Inspections"])
    add_ref(r["id"], RNDT.derivedFrom, r.get("derivedFrom"), IDS["Images"])
    add_lit(r["id"], RNDT.storageLocation, r.get("storageLocation"))
    add_lit(r["id"], RNDT.fileFormat, r.get("fileFormat"))
    add_ref(r["id"], RNDT.viewedWithSoftware, r.get("viewedWithSoftware"), IDS["Software"])
    # 'kind' is informational only and is deliberately not written to the ontology.

for r in rows("Reports"):
    typed(r["id"], "InformationEntity", r.get("label"))
    add_ref(r["id"], RNDT.belongsToInspection, r.get("inspection"), IDS["Inspections"])

for r in rows("Indications"):
    typed(r["id"], "InformationEntity", r.get("label"))
    add_ref(r["id"], RNDT.appliesAcceptanceCriteria, r.get("acceptanceCriteria"), IDS["AcceptanceCriteria"])
    img = r.get("image", "")
    if img and img in IDS["Images"]:
        g.add((U(img), RNDT.hasIndication, U(r["id"])))   # image → indication

for r in rows("ActivityLog"):
    g.add((U(r["id"]), RDF.type, OWL.NamedIndividual))
    add_vocab_ref(r["id"], RNDT.logsActivity, r.get("activity"), ACTIVITY_IRI)
    add_ref(r["id"], RNDT.logForInspection, r.get("inspection"), IDS["Inspections"])
    add_ref(r["id"], RNDT.logPerformedBy, r.get("performedBy"), IDS["Participants"])
    add_lit(r["id"], RNDT.logDate, r.get("date"), XSD.date)
    add_lit(r["id"], RNDT.logResult, r.get("result"))

# ── Split the populated data one file per method ──────────────────────────────
# Which method an inspection belongs to is read off the data, not hardcoded: an
# inspection is DR or UT according to which extension sheet its procedure appears
# on. Anything reached by exactly one method's episode goes in that method's file;
# anything both episodes refer to (shared people, the governing specification) goes
# in the shared file, so neither method file duplicates it or dangles without it.

PROC_METHOD = {r["id"]: "DR" for r in rows("ProceduresDR")}
PROC_METHOD.update({r["id"]: "UT" for r in rows("ProceduresUT")})
INSP_METHOD = {r["id"]: PROC_METHOD.get(r.get("procedure")) for r in rows("Inspections")}

_claims = defaultdict(set)
def claim(i, m):
    if i and m: _claims[i].add(m)

for r in rows("Inspections"):
    m = INSP_METHOD.get(r["id"])
    for col in ("id", "component", "procedure", "acceptanceCriteria",
                "equipment", "specification", "facility"):
        claim(r.get(col), m)
for r in rows("Procedures"):    claim(r.get("approvedBy"), PROC_METHOD.get(r["id"]))
for r in rows("ProceduresDR"):  claim(r["id"], "DR")
for r in rows("ProceduresUT"):  claim(r["id"], "UT")
for r in rows("InspectionsDR"): claim(r["id"], "DR")
for r in rows("InspectionsUT"): claim(r["id"], "UT"); claim(r.get("probe"), "UT")
for r in rows("EquipmentUT"):   claim(r["id"], "UT")

_img_method = {}
for r in rows("Images"):
    m = INSP_METHOD.get(r.get("inspection")); _img_method[r["id"]] = m
    claim(r["id"], m); claim(r.get("viewedWithSoftware"), m)
for r in rows("Indications"):   claim(r["id"], _img_method.get(r.get("image")))
for r in rows("Reports"):       claim(r["id"], INSP_METHOD.get(r.get("inspection")))
for r in rows("ActivityLog"):
    m = INSP_METHOD.get(r.get("inspection"))
    claim(r["id"], m); claim(r.get("performedBy"), m)

def file_for(iri):
    ms = _claims.get(_local(iri), set())
    return next(iter(ms)) if len(ms) == 1 else "shared"

GRAPHS = {}
for _k in TTL_OUT:
    _sg = rdflib.Graph()
    _sg.bind("rndt", RNDT); _sg.bind("rndt-dr", RNDT_DR); _sg.bind("rndt-ut", RNDT_UT)
    _sg.bind("rdfs", RDFS); _sg.bind("owl", OWL); _sg.bind("xsd", XSD)
    GRAPHS[_k] = _sg
for _t in g:
    GRAPHS[file_for(_t[0])].add(_t)
for _k, _path in TTL_OUT.items():
    GRAPHS[_k].serialize(destination=_path, format="turtle")

display(table(["Populated file", "Individuals", "Triples", "Holds"],
              [[TTL_OUT[k],
                len(set(GRAPHS[k].subjects(RDF.type, OWL.NamedIndividual))),
                len(GRAPHS[k]), d]
               for k, d in (("shared", "individuals both methods' episodes refer to"),
                            ("DR", "the Digital Radiography episodes"),
                            ("UT", "the Ultrasonic Testing episode"))]
              + [["— all three —", len(set(g.subjects(RDF.type, OWL.NamedIndividual))), len(g), ""]],
              caption="Conversion complete", classes={0: "mono", 1: "mono", 2: "mono"}))

Populated file,Individuals,Triples,Holds
rndt-populated-shared.ttl,6,26,individuals both methods' episodes refer to
rndt-populated-dr.ttl,212,1291,the Digital Radiography episodes
rndt-populated-ut.ttl,36,211,the Ultrasonic Testing episode
— all three —,254,1528,


In [6]:
# The 34 competency questions: 23 generic ones that hold for any NDE method, plus
# 6 exercising the DR extension and 5 exercising the UT extension. Each is standard
# SPARQL 1.1 and runs against your data merged with the ontology schema.

full = rdflib.Graph()
for _f in SCHEMA_FILES:
    full.parse(_f, format="turtle")
for t in g:
    full.add(t)

P = """PREFIX rndt: <http://www.cranfield.ac.uk/rndt-ontology#>
PREFIX rndt-dr: <http://www.cranfield.ac.uk/rndt-ontology/dr#>
PREFIX rndt-ut: <http://www.cranfield.ac.uk/rndt-ontology/ut#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""

# ── WHICH ONE each question asks about ───────────────────────────────────────
# Every competency question is of the form "...for THIS image / THIS inspection".
# These are the subjects it asks about. Change an id here to re-ask the same
# question of a different image, inspection, component or inspector.
#   rndt:LOCALNAME  -> an individual from your data
#   "text"          -> a literal value
# The generic questions are asked of each method's worked example in turn — same
# SPARQL, different subject. That substitution is the whole method-independence
# claim, so the two parameterisations live side by side rather than in separate cells.
PARAMS_BY_METHOD = {
    "DR": {
         1: ("image",      "rndt:PROC_001"),   2: ("image",      "rndt:PROC_001"),
         3: ("image",      "rndt:PROC_001"),   4: ("inspection", "rndt:INSP_001"),
         5: ("inspection", "rndt:INSP_001"),   6: ("inspection", "rndt:INSP_001"),
         7: ("inspection", "rndt:INSP_001"),   8: ("inspection", "rndt:INSP_001"),
         9: ("inspection", "rndt:INSP_001"),  10: ("image",      "rndt:PROC_001"),
        11: ("component",  "rndt:C_001"),     12: ("report",     "rndt:REPORT_001"),
        13: ("inspection", "rndt:INSP_001"),  14: ("inspection", "rndt:INSP_001"),
        15: ("image",      "rndt:PROC_001"),  16: ("image",      "rndt:PROC_001"),
        17: ("serial",     '"SN00123"'),      18: ("component",  "rndt:C_001"),
        19: ("inspector",  "rndt:P_INS_01"),  20: ("image",      "rndt:PROC_001"),
        21: ("indication", "rndt:IND_001"),   22: ("image",      "rndt:PROC_003"),
        23: ("image",      "rndt:PROC_001"),
    },
    "UT": {
         1: ("image",      "rndt:PROC_010"),   2: ("image",      "rndt:PROC_010"),
         3: ("image",      "rndt:PROC_010"),   4: ("inspection", "rndt:INSP_010"),
         5: ("inspection", "rndt:INSP_010"),   6: ("inspection", "rndt:INSP_010"),
         7: ("inspection", "rndt:INSP_010"),   8: ("inspection", "rndt:INSP_010"),
         9: ("inspection", "rndt:INSP_010"),  10: ("image",      "rndt:PROC_010"),
        11: ("component",  "rndt:C_010"),     12: ("report",     "rndt:REPORT_010"),
        13: ("inspection", "rndt:INSP_010"),  14: ("inspection", "rndt:INSP_010"),
        15: ("image",      "rndt:PROC_010"),  16: ("image",      "rndt:PROC_010"),
        17: ("serial",     '"SN01003"'),      18: ("component",  "rndt:C_010"),
        19: ("inspector",  "rndt:P_INS_01"),  20: ("image",      "rndt:PROC_010"),
        21: ("indication", "rndt:IND_003"),   22: ("image",      "rndt:PROC_010"),
        23: ("image",      "rndt:PROC_010"),
    },
}

# A method-specific question has no "other side" to be asked of, so it carries its
# own subject. INSP_009 is the fully-instrumented DR episode; INSP_010 is the UT one.
PARAMS_SPECIFIC = {
    24: ("inspection", "rndt:INSP_009"),  25: ("inspection", "rndt:INSP_009"),
    27: ("inspection", "rndt:INSP_009"),  28: ("inspection", "rndt:INSP_009"),
    29: ("inspection", "rndt:INSP_009"),  30: ("inspection", "rndt:INSP_009"),
    26: ("inspection", "rndt:INSP_010"),  31: ("inspection", "rndt:INSP_010"),
    32: ("inspection", "rndt:INSP_010"),  33: ("inspection", "rndt:INSP_010"),
    34: ("inspection", "rndt:INSP_010"),
}

# Which questions are generic and which belong to one method. Everything downstream
# — the method-independence table, the coverage table, the per-method totals — is
# derived from this, so a third method's questions drop in by adding entries here.
SCOPE = {**{k: "shared" for k in PARAMS_BY_METHOD["DR"]},
         **{24: "DR", 25: "DR", 27: "DR", 28: "DR", 29: "DR", 30: "DR",
            26: "UT", 31: "UT", 32: "UT", 33: "UT", 34: "UT"}}
SHARED  = sorted(k for k, v in SCOPE.items() if v == "shared")
METHODS = ["DR", "UT"]

# The primary run answers the generic questions of the DR example and each
# method-specific question of its own example.
PARAMS = {**PARAMS_BY_METHOD["DR"], **PARAMS_SPECIFIC}

# Each entry: (question, SPARQL template). {{VALUES}} is filled from PARAMS above.
CQ = {}
CQ[1] = ("Which component is associated with this image?", P + """
SELECT ?componentLabel WHERE {
  {{VALUES}}
  ?image rndt:belongsToInspection ?inspection .
  ?inspection rndt:usesEntity ?component .
  ?component rdfs:label ?componentLabel .
}""")
CQ[2] = ("Who interpreted this image?", P + """
SELECT ?interpreterLabel WHERE {
  {{VALUES}}
  ?image rndt:belongsToInspection ?inspection .
  ?log rndt:logsActivity rndt:IdentifyIndications ;
       rndt:logForInspection ?inspection ;
       rndt:logPerformedBy ?interpreter .
  ?interpreter rdfs:label ?interpreterLabel .
}""")
CQ[3] = ("What certification level did the interpreter hold?", P + """
SELECT ?authorityLabel WHERE {
  {{VALUES}}
  ?image rndt:belongsToInspection ?inspection .
  ?log rndt:logsActivity rndt:IdentifyIndications ;
       rndt:logForInspection ?inspection ;
       rndt:logPerformedBy ?interpreter .
  ?interpreter rndt:memberOfAuthorityClass ?authority .
  ?authority rdfs:label ?authorityLabel .
}""")
CQ[4] = ("Which procedure or work instruction was used for this inspection?", P + """
SELECT ?procedureLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesProcedure ?procedure .
  ?procedure rdfs:label ?procedureLabel .
}""")
CQ[5] = ("Which revision of the procedure was used?", P + """
SELECT ?revision WHERE {
  {{VALUES}}
  ?inspection rndt:usesProcedure ?procedure .
  ?procedure rndt:procedureRevision ?revision .
}""")
CQ[6] = ("Who approved the procedure used for the inspection?", P + """
SELECT ?approverLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesProcedure ?procedure .
  ?procedure rndt:approvedBy ?approver .
  ?approver rdfs:label ?approverLabel .
}""")
CQ[7] = ("What certification level did the procedure approver hold?", P + """
SELECT ?authorityLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesProcedure ?procedure .
  ?procedure rndt:approvedBy ?approver .
  ?approver rndt:memberOfAuthorityClass ?authority .
  ?authority rdfs:label ?authorityLabel .
}""")
CQ[8] = ("Which acceptance criteria were used for this inspection?", P + """
SELECT ?criteriaLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesAcceptanceCriteria ?criteria .
  ?criteria rdfs:label ?criteriaLabel .
}""")
CQ[9] = ("Which equipment was used for the inspection?", P + """
SELECT ?equipmentLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesEquipment ?equipment .
  ?equipment rdfs:label ?equipmentLabel .
}""")
CQ[10] = ("Which software was used to view or interpret the image?", P + """
SELECT ?softwareLabel WHERE {
  {{VALUES}}
  ?image rndt:viewedWithSoftware ?software .
  ?software rdfs:label ?softwareLabel .
}""")
CQ[11] = ("What is the inspection status of this component?", P + """
SELECT ?disposition WHERE {
  {{VALUES}}
  ?inspection rndt:usesEntity ?component ;
              rndt:disposition ?disposition .
}""")
CQ[12] = ("Which inspection produced a particular inspection report?", P + """
SELECT ?inspectionLabel WHERE {
  {{VALUES}}
  ?report rndt:belongsToInspection ?inspection .
  ?inspection rdfs:label ?inspectionLabel .
}""")
CQ[13] = ("Who produced the inspection report?", P + """
SELECT ?producerLabel WHERE {
  {{VALUES}}
  ?log rndt:logsActivity rndt:ApproveInspectionReport ;
       rndt:logForInspection ?inspection ;
       rndt:logPerformedBy ?producer .
  ?producer rdfs:label ?producerLabel .
}""")
CQ[14] = ("Which specification applies to this inspection?", P + """
SELECT ?specLabel WHERE {
  {{VALUES}}
  ?inspection rndt:appliesSpecification ?spec .
  ?spec rdfs:label ?specLabel .
}""")
CQ[15] = ("Where is a particular digital image stored?", P + """
SELECT ?storageLocation WHERE {
  {{VALUES}}
  ?image rndt:storageLocation ?storageLocation .
}""")
CQ[16] = ("What file format is used for a particular image?", P + """
SELECT ?fileFormat WHERE {
  {{VALUES}}
  ?image rndt:fileFormat ?fileFormat .
}""")
CQ[17] = ("Which images are associated with a particular serial number?", P + """
SELECT ?imageLabel WHERE {
  {{VALUES}}
  ?component rndt:serialNumber ?serial .
  ?inspection rndt:usesEntity ?component .
  ?image rndt:belongsToInspection ?inspection ;
         rndt:fileFormat ?format ;
         rdfs:label ?imageLabel .
}""")
CQ[18] = ("Which inspector interpreted images for a particular component?", P + """
SELECT ?inspectorLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesEntity ?component .
  ?log rndt:logsActivity rndt:IdentifyIndications ;
       rndt:logForInspection ?inspection ;
       rndt:logPerformedBy ?inspector .
  ?inspector rdfs:label ?inspectorLabel .
}""")
CQ[19] = ("Which components were interpreted by a particular inspector?", P + """
SELECT DISTINCT ?componentLabel WHERE {
  {{VALUES}}
  ?log rndt:logsActivity rndt:IdentifyIndications ;
       rndt:logForInspection ?inspection ;
       rndt:logPerformedBy ?inspector .
  ?inspection rndt:usesEntity ?component .
  ?component rdfs:label ?componentLabel .
}""")
CQ[20] = ("What inspection decision resulted from the interpretation?", P + """
SELECT ?disposition WHERE {
  {{VALUES}}
  ?image rndt:belongsToInspection ?inspection .
  ?inspection rndt:disposition ?disposition .
}""")
CQ[21] = ("Which acceptance criteria were applied to a particular indication?", P + """
SELECT ?criteriaLabel WHERE {
  {{VALUES}}
  ?indication rndt:appliesAcceptanceCriteria ?criteria .
  ?criteria rdfs:label ?criteriaLabel .
}""")
CQ[22] = ("What indications were identified in this image?", P + """
SELECT ?indicationLabel WHERE {
  {{VALUES}}
  ?image rndt:hasIndication ?indication .
  ?indication rdfs:label ?indicationLabel .
}""")
CQ[23] = ("Which facility interpreted the image?", P + """
SELECT ?facilityLabel WHERE {
  {{VALUES}}
  ?image rndt:belongsToInspection ?inspection .
  ?inspection rndt:performedAtFacility ?facility .
  ?facility rdfs:label ?facilityLabel .
}""")
CQ[24] = ("What IQI type and radiographic technique were required by the procedure used for this inspection?", P + """
SELECT ?iqiLabel ?techniqueLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesProcedure ?procedure .
  ?procedure rndt-dr:usesIQIType ?iqi ;
             rndt-dr:usesRadiographicTechnique ?technique .
  ?iqi rdfs:label ?iqiLabel .
  ?technique rdfs:label ?techniqueLabel .
}""")
CQ[25] = ("What IQI sensitivity was achieved for this inspection?", P + """
SELECT ?sensitivity WHERE {
  {{VALUES}}
  ?inspection rndt-dr:iqiSensitivityAchieved ?sensitivity .
}""")
CQ[26] = ("What UT method was required by the procedure used for this inspection?", P + """
SELECT ?methodLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesProcedure ?procedure .
  ?procedure rndt-ut:usesUTMethod ?method .
  ?method rdfs:label ?methodLabel .
}""")

# ── DR extension questions ───────────────────────────────────────────────────
CQ[27] = ("What source-to-detector and source-to-object distances did the approved procedure specify?", P + """
SELECT ?sdd ?sod WHERE {
  {{VALUES}}
  ?inspection rndt:usesProcedure ?procedure .
  ?procedure rndt-dr:sourceToDetectorDistance ?sdd ;
             rndt-dr:sourceToObjectDistance ?sod .
}""")
CQ[28] = ("Which DR-specific setup activities were logged for this inspection, and by whom?", P + """
SELECT ?activityLabel ?personLabel WHERE {
  {{VALUES}}
  ?log rndt:logForInspection ?inspection ;
       rndt:logsActivity ?activity ;
       rndt:logPerformedBy ?person .
  ?activity rdfs:label ?activityLabel .
  ?person rdfs:label ?personLabel .
  FILTER(STRSTARTS(STR(?activity), "http://www.cranfield.ac.uk/rndt-ontology/dr#"))
}""")
CQ[29] = ("Was IQI placement verified for this inspection, and with what result?", P + """
SELECT ?personLabel ?date ?result WHERE {
  {{VALUES}}
  ?log rndt:logsActivity rndt-dr:VerifyIQIPlacement ;
       rndt:logForInspection ?inspection ;
       rndt:logPerformedBy ?person .
  ?person rdfs:label ?personLabel .
  OPTIONAL { ?log rndt:logDate ?date }
  OPTIONAL { ?log rndt:logResult ?result }
}""")
CQ[30] = ("Did the achieved IQI sensitivity meet the sensitivity the procedure required?", P + """
SELECT ?achieved ?required ?verdict WHERE {
  {{VALUES}}
  ?inspection rndt-dr:iqiSensitivityAchieved ?achieved ;
              rndt:usesProcedure ?procedure .
  ?procedure rndt-dr:requiredIQISensitivity ?required .
  BIND(IF(?achieved <= ?required, "Pass", "Fail") AS ?verdict)
}""")

# ── UT extension questions ───────────────────────────────────────────────────
CQ[31] = ("Which probe, of what frequency, diameter and serial number, was used for this inspection?", P + """
SELECT ?probeLabel ?serialNumber ?frequencyMHz ?diameterMM WHERE {
  {{VALUES}}
  ?inspection rndt-ut:usesProbe ?probe .
  ?probe rdfs:label ?probeLabel .
  OPTIONAL { ?probe rndt:serialNumber ?serialNumber }
  OPTIONAL { ?probe rndt-ut:probeFrequency ?frequencyMHz }
  OPTIONAL { ?probe rndt-ut:probeDiameter ?diameterMM }
}""")
CQ[32] = ("Which UT-specific setup activities were logged for this inspection, and by whom?", P + """
SELECT ?activityLabel ?personLabel WHERE {
  {{VALUES}}
  ?log rndt:logForInspection ?inspection ;
       rndt:logsActivity ?activity ;
       rndt:logPerformedBy ?person .
  ?activity rdfs:label ?activityLabel .
  ?person rdfs:label ?personLabel .
  FILTER(STRSTARTS(STR(?activity), "http://www.cranfield.ac.uk/rndt-ontology/ut#"))
}""")
CQ[33] = ("Was the calibration verified as still valid across the scan, and with what result?", P + """
SELECT ?personLabel ?date ?result WHERE {
  {{VALUES}}
  ?log rndt:logsActivity rndt-ut:VerifyCalibrationVerification ;
       rndt:logForInspection ?inspection ;
       rndt:logPerformedBy ?person .
  ?person rdfs:label ?personLabel .
  OPTIONAL { ?log rndt:logDate ?date }
  OPTIONAL { ?log rndt:logResult ?result }
}""")
CQ[34] = ("What UT method and beam angle did this inspection use?", P + """
SELECT ?methodLabel ?beamAngleDegrees WHERE {
  {{VALUES}}
  ?inspection rndt:usesProcedure ?procedure ;
              rndt-ut:usesProbe ?probe .
  ?procedure rndt-ut:usesUTMethod ?method .
  ?method rdfs:label ?methodLabel .
  ?probe rndt-ut:probeAngle ?beamAngleDegrees .
}""")

def sparql_for(k):
    """Fill the {{VALUES}} placeholder from PARAMS."""
    var, term = PARAMS[k]
    return CQ[k][1].replace("{{VALUES}}", f"VALUES ?{var} {{ {term} }}")

def subject_of(k):
    """Human-readable description of what this question was asked about."""
    var, term = PARAMS[k]
    if term.startswith('"'):
        return var, term.strip('"'), ""
    local = term.split(":")[-1]
    lbl = full.value(RNDT[local], RDFS.label)
    if lbl is None:
        return var, local, "not in your data"
    return var, local, str(lbl)

RESULTS = {}
for k in sorted(CQ):
    try:
        RESULTS[k] = ("ok", [", ".join(str(x) for x in row) for row in full.query(sparql_for(k))])
    except Exception as e:
        RESULTS[k] = ("error", [str(e)])

_answered = sum(1 for k, (st, rows_) in RESULTS.items() if st == "ok" and rows_)
_rows = []
for k in sorted(CQ):
    st, res = RESULTS[k]
    if st == "error":
        badge, cls = "ERROR", "bad"
    elif res:
        badge, cls = "answered", "ok"
    else:
        badge, cls = "no data", "warn"
    preview = "; ".join(res[:2]) + ("  …" if len(res) > 2 else "")
    var, sid, slbl = subject_of(k)
    asked = (f'<span class="mono">?{var} = {html.escape(sid)}</span>'
             + (f'<br><span class="dim">{html.escape(slbl)}</span>' if slbl else ""))
    _rows.append([f"CQ-{k:02d}", CQ[k][0], asked, f'<span class="{cls}">{badge}</span>',
                  len(res) if st == "ok" else "—", preview])

display(table(["#", "Competency question", "Asked about", "Result", "Rows", "Answer"], _rows,
              caption=f"{_answered} of {len(CQ)} questions answered by this data",
              classes={0: "mono", 2: "raw", 3: "raw", 4: "mono"}))

#,Competency question,Asked about,Result,Rows,Answer
CQ-01,Which component is associated with this image?,?image = PROC_001Processed radiographic image (INSP_001),answered,1,Turbine Blade Casting
CQ-02,Who interpreted this image?,?image = PROC_001Processed radiographic image (INSP_001),answered,1,David Whitfield
CQ-03,What certification level did the interpreter hold?,?image = PROC_001Processed radiographic image (INSP_001),answered,1,Level 2 Technical Evaluation Authority
CQ-04,Which procedure or work instruction was used for this inspection?,?inspection = INSP_001Digital Radiography inspection INSP_001,answered,1,Remote digital radiography procedure RNDT-DR-001
CQ-05,Which revision of the procedure was used?,?inspection = INSP_001Digital Radiography inspection INSP_001,answered,1,Rev C
CQ-06,Who approved the procedure used for the inspection?,?inspection = INSP_001Digital Radiography inspection INSP_001,answered,1,Thomas Okafor
CQ-07,What certification level did the procedure approver hold?,?inspection = INSP_001Digital Radiography inspection INSP_001,answered,1,Level 3 Governance and Assurance Authority
CQ-08,Which acceptance criteria were used for this inspection?,?inspection = INSP_001Digital Radiography inspection INSP_001,answered,1,Castings acceptance criteria ACC-CAST-B
CQ-09,Which equipment was used for the inspection?,?inspection = INSP_001Digital Radiography inspection INSP_001,answered,1,Digital radiography system DR-4500
CQ-10,Which software was used to view or interpret the image?,?image = PROC_001Processed radiographic image (INSP_001),answered,1,DICONDE image viewer v4.2


In [7]:
# Method-independence check: re-run the generic questions against a second example —
# the UT inspection (INSP_010) — instead of the DR one (INSP_001), with zero SPARQL
# changes, and put the two side by side. The method-specific questions are excluded:
# they are method-specific by design (DR's IQI/technique/sensitivity/geometry, UT's
# method/probe/calibration), so there is no "other side" to compare them against.

def sparql_with(k, params):
    var, term = params[k]
    return CQ[k][1].replace("{{VALUES}}", f"VALUES ?{var} {{ {term} }}")

def run(k, params):
    try:
        return ("ok", [", ".join(str(x) for x in row) for row in full.query(sparql_with(k, params))])
    except Exception as e:
        return ("error", [str(e)])

# The generic questions, answered once per method.
SHARED_RESULTS = {m: {k: run(k, PARAMS_BY_METHOD[m]) for k in SHARED} for m in METHODS}

def _answer_cell(st, res):
    if st == "error":
        return f'<span class="bad">ERROR: {html.escape(res[0])}</span>'
    if res:
        preview = "; ".join(res[:2]) + ("  \u2026" if len(res) > 2 else "")
        return html.escape(preview)
    return '<span class="warn">no data</span>'

_mi_rows = [[f"CQ-{k:02d}", CQ[k][0]] + [_answer_cell(*SHARED_RESULTS[m][k]) for m in METHODS]
            for k in SHARED]
_ok = {m: sum(1 for k in SHARED if SHARED_RESULTS[m][k][0] == "ok" and SHARED_RESULTS[m][k][1])
       for m in METHODS}

display(table(["#", "Competency question", "DR example (INSP_001)", "UT example (INSP_010)"], _mi_rows,
              caption="Method independence \u2014 the same "
                      f"{len(SHARED)} generic questions, unchanged SPARQL, two methods: "
                      + " \u00b7 ".join(f"{m} {_ok[m]}/{len(SHARED)}" for m in METHODS),
              classes={2: "raw", 3: "raw"}))


#,Competency question,DR example (INSP_001),UT example (INSP_010)
CQ-01,Which component is associated with this image?,Turbine Blade Casting,Compressor Disk Forging
CQ-02,Who interpreted this image?,David Whitfield,David Whitfield
CQ-03,What certification level did the interpreter hold?,Level 2 Technical Evaluation Authority,Level 2 Technical Evaluation Authority
CQ-04,Which procedure or work instruction was used for this inspection?,Remote digital radiography procedure RNDT-DR-001,Remote UT disk inspection procedure RNDT-UT-001
CQ-05,Which revision of the procedure was used?,Rev C,Rev A
CQ-06,Who approved the procedure used for the inspection?,Thomas Okafor,Thomas Okafor
CQ-07,What certification level did the procedure approver hold?,Level 3 Governance and Assurance Authority,Level 3 Governance and Assurance Authority
CQ-08,Which acceptance criteria were used for this inspection?,Castings acceptance criteria ACC-CAST-B,Disk UT acceptance criteria ACC-UT-DISK-A
CQ-09,Which equipment was used for the inspection?,Digital radiography system DR-4500,Phased array flaw detector PA-3200
CQ-10,Which software was used to view or interpret the image?,DICONDE image viewer v4.2,UT analysis suite TomoView-class v3.0


In [8]:
# Full coverage table \u2014 every question, one column per method. The generic questions
# take both columns from the method-independence check above. A method-specific
# question has no counterpart property on the other method, so the other column reads
# "n/a" rather than counting as a failure. Scope comes from the SCOPE table, so this
# stays correct as questions are added.

_cov_rows, _totals = [], {m: 0 for m in METHODS}
for k in sorted(CQ):
    scope = SCOPE[k]
    cells = []
    for m in METHODS:
        if scope == "shared":
            st, res = SHARED_RESULTS[m][k]
        elif scope == m:
            st, res = RESULTS[k]
        else:
            cells.append(f'<span class="dim">n/a \u2014 {scope}-specific property</span>')
            continue
        cells.append(_answer_cell(st, res))
        if st == "ok" and res:
            _totals[m] += 1
    _cov_rows.append([f"CQ-{k:02d}", CQ[k][0], "shared" if scope == "shared" else f"{scope}-only"] + cells)

_asked = {m: sum(1 for k in CQ if SCOPE[k] in ("shared", m)) for m in METHODS}

display(table(["#", "Competency question", "Scope"] + METHODS, _cov_rows,
              caption=f"All {len(CQ)} CQs \u2014 "
                      + " \u00b7 ".join(f"{m} answers {_totals[m]}/{_asked[m]} of the questions in scope for it"
                                    for m in METHODS),
              classes={2: "mono", 3: "raw", 4: "raw"}))


#,Competency question,Scope,DR,UT
CQ-01,Which component is associated with this image?,shared,Turbine Blade Casting,Compressor Disk Forging
CQ-02,Who interpreted this image?,shared,David Whitfield,David Whitfield
CQ-03,What certification level did the interpreter hold?,shared,Level 2 Technical Evaluation Authority,Level 2 Technical Evaluation Authority
CQ-04,Which procedure or work instruction was used for this inspection?,shared,Remote digital radiography procedure RNDT-DR-001,Remote UT disk inspection procedure RNDT-UT-001
CQ-05,Which revision of the procedure was used?,shared,Rev C,Rev A
CQ-06,Who approved the procedure used for the inspection?,shared,Thomas Okafor,Thomas Okafor
CQ-07,What certification level did the procedure approver hold?,shared,Level 3 Governance and Assurance Authority,Level 3 Governance and Assurance Authority
CQ-08,Which acceptance criteria were used for this inspection?,shared,Castings acceptance criteria ACC-CAST-B,Disk UT acceptance criteria ACC-UT-DISK-A
CQ-09,Which equipment was used for the inspection?,shared,Digital radiography system DR-4500,Phased array flaw detector PA-3200
CQ-10,Which software was used to view or interpret the image?,shared,DICONDE image viewer v4.2,UT analysis suite TomoView-class v3.0


In [9]:
# Full detail — every question with its SPARQL and complete results.
# Useful as documentation of the queries, and for debugging a question that returned nothing.

_out = []
for k in sorted(CQ):
    q = CQ[k][0]
    sparql = sparql_for(k)
    var, sid, slbl = subject_of(k)
    st, res = RESULTS[k]
    if st == "error":
        body = f'<div class="bad">ERROR: {html.escape(res[0])}</div>'
    elif res:
        body = "<ul>" + "".join(f"<li>{html.escape(r)}</li>" for r in res) + "</ul>"
    else:
        body = ('<div class="warn">No rows — the data does not answer this question. '
                'Check the sheets listed for it in the CQ Coverage tab of the template.</div>')
    _out.append(
        f'<div style="margin:0 0 20px"><div class="cap">CQ-{k:02d}</div>'
        f'<div style="font-weight:600;margin-bottom:4px">{html.escape(q)}</div>'
        f'<div class="note">Asked about <span class="mono">?{var} = {html.escape(sid)}</span>'
        f'{" &middot; " + html.escape(slbl) if slbl else ""}</div>'
        f'<pre>{html.escape(sparql.strip())}</pre>{body}</div>')

display(HTML(_CSS + '<div class="rndt">' + "".join(_out) + "</div>"))

In [10]:
# Regenerate the knowledge-graph visualisations from the schema files.
# The node/edge data is embedded in each HTML page, so it used to be hand-maintained
# and drifted: the DR page claimed 2 individuals long after the module had grown to
# 11. Rebuilding it here from the .ttl means the pages cannot disagree with the
# ontology they illustrate.

KG_TYPES = ["ProcessStage", "MajorAction", "ActionArea", "Activity", "Capability",
            "RequirementConcept", "InformationEntity", "AuthorityClass", "Participant"]
# Controlled-vocabulary classes (IQIType, RadiographicTechnique, UTMethod) and the
# instance-level classes (PhysicalEntity, Probe, Software, Facility, Procedure) are
# deliberately absent: these pages show the process taxonomy, not the populated data.

KG_PAGES = [("rndt-core-knowledge-graph.html", ["rndt-core.ttl"],               None),
            ("rndt-dr-knowledge-graph.html",   ["rndt-core.ttl", "rndt-dr.ttl"], "dr"),
            ("rndt-ut-knowledge-graph.html",   ["rndt-core.ttl", "rndt-ut.ttl"], "ut")]

def _kg_module(f): return "core" if "core" in f else ("dr" if "-dr" in f else "ut")

def kg_build(files):
    graphs = [(f, rdflib.Graph().parse(f, format="turtle")) for f in files]
    merged = rdflib.Graph()
    for _, gr in graphs:
        for t in gr: merged.add(t)
    props = set(merged.subjects(RDF.type, OWL.ObjectProperty))
    nodes = {}
    for f, gr in graphs:
        for cls in KG_TYPES:
            for s in gr.subjects(RDF.type, RNDT[cls]):
                nodes.setdefault(_local(s), {
                    "id": _local(s), "label": str(merged.value(s, RDFS.label) or _local(s)),
                    "type": cls, "termId": str(merged.value(s, RNDT.termId) or ""),
                    "module": _kg_module(f)})
    edges = [{"source": _local(s), "target": _local(o), "rel": _local(pr), "module": _kg_module(f)}
             for f, gr in graphs for s, pr, o in gr
             if pr in props and _local(s) in nodes and _local(o) in nodes]
    return ([nodes[k] for k in sorted(nodes)],
            sorted(edges, key=lambda e: (e["source"], e["rel"], e["target"])))

_kg_rows = []
for page, files, module in KG_PAGES:
    nodes, edges = kg_build(files)
    src = Path(page).read_text()
    blob = json.dumps({"nodes": nodes, "edges": edges}, separators=(",", ":"))
    src, n = re.subn(r"const data = \{.*?\};", "const data = " + blob.replace("\\", "\\\\") + ";",
                     src, count=1, flags=re.S)
    assert n == 1, f"{page}: could not find the embedded data blob"

    if module is None:
        cap = (f"{len(nodes)} individuals &middot; {len(edges)} relationships &middot; "
               f"{len({x['type'] for x in nodes})} populated classes &middot; method-agnostic")
        added = len(nodes)
    else:
        own_n = [x for x in nodes if x["module"] == module]
        own_e = [x for x in edges if x["module"] == module]
        added = len(own_n)
        cap = (f"{len(own_n)} {module.upper()} individuals &middot; {len(own_e)} {module.upper()} edges "
               f"&middot; overlaid on {len(nodes) - len(own_n)} shared core individuals")
        counts = {}
        for x in own_n: counts[x["type"]] = counts.get(x["type"], 0) + 1
        breakdown = " + ".join(f"{v} <code>{k}</code>" for k, v in sorted(counts.items(), key=lambda kv: -kv[1]))
        src = re.sub(r"adds only <b>\d+ individuals</b> \([^)]*\) and <b>\d+ wiring edges</b>",
                     f"adds only <b>{len(own_n)} individuals</b> ({breakdown}) "
                     f"and <b>{len(own_e)} wiring edges</b>", src, count=1)
    src = re.sub(r'(class="kg-sub">)[^<]*(<)', lambda m: m.group(1) + cap + m.group(2), src, count=1)
    Path(page).write_text(src)
    _kg_rows.append([page, len(nodes), len(edges), added])

display(table(["Page", "Nodes drawn", "Edges drawn", "Contributed by this module"], _kg_rows,
              caption="Knowledge-graph pages regenerated from the schema files",
              classes={0: "mono", 1: "mono", 2: "mono", 3: "mono"}))


Page,Nodes drawn,Edges drawn,Contributed by this module
rndt-core-knowledge-graph.html,102,122,102
rndt-dr-knowledge-graph.html,113,140,11
rndt-ut-knowledge-graph.html,110,136,8
